In [ ]:
from google.colab import files

uploaded = files.upload()

Saving skin_ai_dataset.zip to skin_ai_dataset.zip


In [ ]:
import zipfile
import os

zip_path = "/content/skin_ai_dataset.zip"
extract_path = "/content/skin_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))

Dataset extracted successfully!
['train', 'val', 'test']


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/skin_dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/skin_dataset/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/skin_dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Classes:", class_names)

Found 7900 files belonging to 6 classes.
Found 1693 files belonging to 6 classes.
Found 1694 files belonging to 6 classes.
Classes: ['acne', 'eczema', 'fungal', 'melanoma', 'psoriasis', 'vitiligo']


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import tensorflow as tf

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 28s 75ms/step - accuracy: 0.5535 - loss: 1.1890 - val_accuracy: 0.6887 - val_loss: 0.7446
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 16s 67ms/step - accuracy: 0.6695 - loss: 0.8187 - val_accuracy: 0.7135 - val_loss: 0.6747
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 21s 67ms/step - accuracy: 0.6995 - loss: 0.7417 - val_accuracy: 0.7395 - val_loss: 0.6311
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - accuracy: 0.7171 - loss: 0.6806 - val_accuracy: 0.7490 - val_loss: 0.6127
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - accuracy: 0.7277 - loss: 0.6609 - val_accuracy: 0.7649 - val_loss: 0.5958
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - accuracy: 0.7348 - loss: 0.6375 - val_accuracy: 0.7566 - val_loss: 0.5937
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - accuracy: 0.7434 - loss: 0.6245 - val_accuracy: 0.7549 - val_loss: 0.5838
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 20s 69ms/step - accuracy: 0.7459 - loss: 0.6078 - 

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Accuracy:", test_accuracy)

53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.7574 - loss: 0.5736
Test Accuracy: 0.7573789954185486


In [ ]:
with open("labels.txt", "w") as f:
    for label in class_names:
        f.write(label + "\n")

print("labels.txt saved")

labels.txt saved


In [ ]:
model.save("skin_model.keras")

print("Keras model saved")

Keras model saved


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("skin_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved")

Saved artifact at '/tmp/tmpn1eocww3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  137501374518032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374519952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374519760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374518608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374520528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374519568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374521104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374522064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374521680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137501374520144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1375013745

In [ ]:
model.save("skin_model.keras")

from google.colab import files
files.download("skin_model.keras")
files.download("labels.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>